In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 117.9 MB/s eta 0:00:00


In [ ]:
import os                                                                        #isletim sistemi ile ilgili bir kutuphane, diskteki dosyalarda dolasabilmemizi sagliyor.
import fitz                                                                      # pdfteki sayfalari metin haline getirebilmek icin gerekli kutuphane.
import pandas as pd                                                              # tablolama icin kullandigimiz kutuphane.  veri setini metin - departman seklinde tablolayacagiz.

from sklearn.model_selection import train_test_split                             # veriyi egitim ve test olarak bolebilmek icin gerekli kutuphane. ben sonrasinda cross validation da deneyecegim.
from sklearn.feature_extraction.text import TfidfVectorizer                      # metni sayisal vektore ceviren kutuphane. yani metni makinnein anlayabilecegi forma ceviriyor
from sklearn.linear_model import LogisticRegression                              # logisticregresyon modeli ile egitecegiz
from sklearn.pipeline import Pipeline                                            # bu birden fazla adimi tek modelde kullanmamizi saglar. basta metni sayisal vektore cevirdik ve logistic regression modelini de kullanmak istedigimiz icin bu adim gerekli.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix              # modelin basarisini olcen metrikleri gorebilmem icin gerekli

import joblib
import re
                                                           # modeli kaydetmek ve tekrar yuklemek icin

In [ ]:
from google.colab import drive                             # drivedan baglantisi kurarak veri setini yukledim

drive.mount('/content/drive')

DATASET_ROOT = "/content/drive/MyDrive/Colab Notebooks/zor_veri_seti"


import os

print("Veri setindeki departman klasörleri:")
print(os.listdir(DATASET_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Veri setindeki departman klasörleri:
['hukuk', 'finans', 'ar_ge', 'insan_kaynaklari', 'musteri_hizmetleri']


In [ ]:
def extract_text_from_pdf(pdf_path):
    try:
        doc = fitz.open(pdf_path)
        full_text = []

        for page in doc:
            text = page.get_text()
            full_text.append(text)

        doc.close()
        text = "\n".join(full_text)


        text = re.sub(r"Sayfa\s*\d+(\s*/\s*\d+)?", " ", text)
        text = re.sub(r"\n{2,}", "\n", text)
        text = re.sub(r"\b\w{1,2}\b", " ", text)   # 1-2 harfli kelimeleri sil
        text = re.sub(r"\s+", " ", text)

        return text.strip()

    except Exception as e:
        print(f"PDF okunamadı: {pdf_path} → Hata: {e}")
        return ""

In [ ]:
test_pdf = "/content/drive/MyDrive/veri_seti/ar_ge/arge1 (1).pdf"       # fonksiyonun dogru calisip calismadigini test ettim.
print(extract_text_from_pdf(test_pdf)[:500])

PDF okunamadı: /content/drive/MyDrive/veri_seti/ar_ge/arge1 (1).pdf → Hata: no such file: '/content/drive/MyDrive/veri_seti/ar_ge/arge1 (1).pdf'



In [ ]:
def load_dataset_from_folders(root_dir):    # bu fonksiyon her klasoru bir departman etiketi olarak alir ve tum pdfleri text label ve path seklinde tabloya ekler

    texts = []                              # burada text label ve path icin birer liste olusturduk
    labels = []
    paths = []


    for label_name in os.listdir(root_dir):                       # ana klasor altindaki departman klasorlerini tek tek dolas. yani bu 5 adet departman klasoru icin olusturdugum for dongusu.
        class_dir = os.path.join(root_dir, label_name)

        if not os.path.isdir(class_dir):                          # dosya ise atliyor cunku etiketleri alabilmemiz icin sadece klasorlere bakmasi gerekiyor su an
            continue


        for pdf_file in os.listdir(class_dir):                    # burada 5 adet departman klasorunun her birinin icindeki pdfler icin olusturdugum for dongusu. yani klasorlerin icinde dolasacak

            if not pdf_file.lower().endswith(".pdf"):             #pdf olmayan dosyalari atlar normalde hepsini pdfe cevirdim ama garanti olmasi icin
                continue

            pdf_path = os.path.join(class_dir, pdf_file)          # o an for dongusunun ustunde oldugu dosya bilgilerini alir bu dosya ile islem yapacagiz


            text = extract_text_from_pdf(pdf_path)                # pdfi metne cevirebilmek icin yazdigimiz fonksiyonu kullandik

            if len(text.strip()) == 0:                            # pdf bos ise atla
                continue

            texts.append(text)                                    # burada okudugu pdfin test label ve path bilgilerini en basta olustuirdugumuz listeye ekliyor.
            labels.append(label_name)
            paths.append(pdf_path)


    df = pd.DataFrame({                                         # tabloyu olustur
        "text": texts,
        "label": labels,
        "path": paths
    })

    return df                                                   # tabloyu dondur

In [ ]:
df = load_dataset_from_folders(DATASET_ROOT)        # tabloyu olusturabiliyor mu diye test kodu
df.head()

,text,label,path
0,"KONU: Hukuk Birimi Değerlendirme, Müşteri Hizm...",hukuk,/content/drive/MyDrive/Colab Notebooks/zor_ver...
1,"KONU: Hukuk Birimi Değerlendirme, Müşteri Hizm...",hukuk,/content/drive/MyDrive/Colab Notebooks/zor_ver...
2,"KONU: Hukuk Birimi Değerlendirme, Müşteri Hizm...",hukuk,/content/drive/MyDrive/Colab Notebooks/zor_ver...
3,"KONU: Hukuk Birimi Değerlendirme, Müşteri Hizm...",hukuk,/content/drive/MyDrive/Colab Notebooks/zor_ver...
4,"KONU: Hukuk Birimi Değerlendirme, Müşteri Hizm...",hukuk,/content/drive/MyDrive/Colab Notebooks/zor_ver...


In [ ]:
X = df["text"]      # Belgelerin metinleri
y = df["label"]     # Hangi departman (IK, Finans, Hukuk, MH, ArGe)

# veriyi eğitim ve test olarak böleceğiz
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # verinin %20'si test, %80'i eğitim
    random_state=42,     # sonuçları tekrar üretilebilir yapmak için
    stratify=y           # her departmandan orantılı örnek gelsin
)

print("Toplam örnek sayısı:", len(df))
print("Eğitim seti sayısı:", len(X_train))
print("Test seti sayısı:", len(X_test))

print("\nEğitim seti sınıf dağılımı:")
print(y_train.value_counts())

print("\nTest seti sınıf dağılımı:")
print(y_test.value_counts())

Toplam örnek sayısı: 3609
Eğitim seti sayısı: 2887
Test seti sayısı: 722

Eğitim seti sınıf dağılımı:
label
ar_ge                 672
finans                560
musteri_hizmetleri    552
insan_kaynaklari      552
hukuk                 551
Name: count, dtype: int64

Test seti sınıf dağılımı:
label
ar_ge                 168
finans                140
hukuk                 138
insan_kaynaklari      138
musteri_hizmetleri    138
Name: count, dtype: int64


In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=30000,        # düşürüldü
        ngram_range=(1, 2),
        min_df=3,                  # EN AZ 3 belgede geçsin
        max_df=0.9,                # çok genel kelimeleri bastır
        analyzer="word",
        lowercase=True
    )),
    ("logreg", LogisticRegression(
        max_iter=2000,
        n_jobs=-1,
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42            # stabilite
    ))
])

print("Model eğitiliyor...")
model.fit(X_train, y_train)
print("Eğitim tamamlandı.")

print("Gerçekleşen iterasyon sayısı:")
print(model.named_steps["logreg"].n_iter_)


Model eğitiliyor...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Eğitim tamamlandı.
Gerçekleşen iterasyon sayısı:
[25]


In [ ]:
y_pred = model.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)



print(classification_report(y_test, y_pred))


cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
print("\nConfusion Matrix:\n")
print(cm)

Accuracy: 1.0
                    precision    recall  f1-score   support

             ar_ge       1.00      1.00      1.00       168
            finans       1.00      1.00      1.00       140
             hukuk       1.00      1.00      1.00       138
  insan_kaynaklari       1.00      1.00      1.00       138
musteri_hizmetleri       1.00      1.00      1.00       138

          accuracy                           1.00       722
         macro avg       1.00      1.00      1.00       722
      weighted avg       1.00      1.00      1.00       722


Confusion Matrix:

[[168   0   0   0   0]
 [  0 140   0   0   0]
 [  0   0 138   0   0]
 [  0   0   0 138   0]
 [  0   0   0   0 138]]


In [ ]:
joblib.dump(model, "departman_modeli.joblib")       # modeli kaydediyorum. makine ogrenmesi modellerinde save() yokmus.

['departman_modeli.joblib']

In [ ]:
print("Toplam örnek sayısı:", len(df))
print(df["label"].value_counts())

Toplam örnek sayısı: 3609
label
ar_ge                 840
finans                700
insan_kaynaklari      690
musteri_hizmetleri    690
hukuk                 689
Name: count, dtype: int64


In [ ]:
def predict_pdf_department_proba(model, pdf_path):    # pdf dosyasini alacak ve her sinif icin bir olasilik dondurecek


    text = extract_text_from_pdf(pdf_path)  # pdften metni cekecek ve text degiskenine atayacak


    if not text.strip():          #metin bossa yani pdf dosyasini okudu texte bir sey cekemedi burada uyari verecek. strip bastaki ve sondaki bosluklari siliyor
        print(f"Uyarı: PDF'ten metin okunamadı -> {pdf_path}")
        return None


    proba = model.predict_proba([text])[0]   #softmax ciktilarini alir yani hangi sinifa ne olasilikla ait. burada tek dosya yukleyip tek bir cevap alacagimiz icin 0 dedik yani ilk ve tek vektoru alacagiz


    class_names = model.classes_            # örn: ['ar_ge', 'finans', 'hukuk', 'insan_kaynaklari', 'musteri_hizmetleri'] yani sinif isimlerini yazcaak


    prob_dict = {cls: float(p) for cls, p in zip(class_names, proba)}    # sinif - olasilik degeri


    prob_dict_sorted = dict(sorted(prob_dict.items(), key=lambda x: x[1], reverse=True))  # 5) buyukten kucuge sırala  yani en yuksek olasilik en ustte olsun

    return prob_dict_sorted   # prob_dict ile sinif-olasilik olarak eslemistik bunu da buyukten kucuge siralayarak dondurur



# Test kodu
example_pdf_path = "/content/drive/MyDrive/Yesil Sertifika Sistemi (YeS-TR).pdf"

print("/content/drive/MyDrive/Yesil Sertifika Sistemi (YeS-TR).pdf", example_pdf_path)
probs = predict_pdf_department_proba(model, example_pdf_path)

print("\nBu belge için departman olasılıkları:")
for dept, p in probs.items():
    print(f"{dept:20s} -> %{p*100:.2f}")

/content/drive/MyDrive/Yesil Sertifika Sistemi (YeS-TR).pdf /content/drive/MyDrive/Yesil Sertifika Sistemi (YeS-TR).pdf

Bu belge için departman olasılıkları:
hukuk                -> %41.62
insan_kaynaklari     -> %25.95
musteri_hizmetleri   -> %17.49
finans               -> %7.83
ar_ge                -> %7.11
